# Notebook 03: nn.Module Patterns

## Why This Matters

The `nn.Module` class is the backbone of every PyTorch model. When engineers miss the registration mechanism -- using a plain Python list instead of `ModuleList`, or a plain tensor instead of `register_buffer` -- parameters silently fail to appear in `model.parameters()`, never reach the GPU when you call `.to(device)`, and never appear in `state_dict()`. These bugs cause hard-to-trace issues: optimizers skip some parameters, `.cuda()` leaves activations on CPU, and checkpoints load incorrectly. Knowing the module machinery deeply prevents an entire class of silent correctness bugs.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
print("Ready.")

## 1. __setattr__ Magic -- How Parameters Get Registered

When you assign an `nn.Parameter` or another `nn.Module` as an attribute of an `nn.Module`, PyTorch's `__setattr__` intercepts the assignment and registers it in internal dicts (`_parameters`, `_modules`, `_buffers`). This registration is what makes `.parameters()`, `.to(device)`, and `state_dict()` work recursively across the entire model tree.

If you assign a plain `torch.Tensor` (not wrapped in `nn.Parameter`), it is stored as a regular Python attribute, **invisible** to the module machinery.

In [ ]:
class RegistrationDemo(nn.Module):
    def __init__(self):
        super().__init__()
        # REGISTERED as a parameter
        self.weight = nn.Parameter(torch.randn(4, 4))
        # REGISTERED as a submodule
        self.linear = nn.Linear(4, 4)
        # NOT registered -- plain tensor stored as regular attribute
        self.unregistered = torch.randn(4, 4)

    def forward(self, x):
        return self.linear(x)

model = RegistrationDemo()
print("Named parameters:")
for name, p in model.named_parameters():
    print(f"  {name}: {p.shape}")

print("\nNamed buffers: (none yet)")
for name, b in model.named_buffers():
    print(f"  {name}: {b.shape}")

print(f"\nunregistered visible to .parameters(): {
    any(p.data_ptr() == model.unregistered.data_ptr() for p in model.parameters())
}")

# state_dict check
sd = model.state_dict()
print(f"\nstate_dict keys: {list(sd.keys())}")
print("Note: unregistered tensor is NOT in state_dict -- it would not survive checkpointing")

## 2. ModuleList vs Plain List -- The Classic Bug

This is probably the most common `nn.Module` bug in production code. If you store submodules in a plain Python list, they are not registered. Calling `.to(device)` on the parent will leave those layers on CPU. The optimizer will not include their parameters. Checkpointing will silently drop them.

In [ ]:
class BuggyModel(nn.Module):
    def __init__(self, n_layers=3):
        super().__init__()
        # BUG: plain list -- submodules are NOT registered
        self.layers = [nn.Linear(8, 8) for _ in range(n_layers)]
        self.out = nn.Linear(8, 2)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x).relu()
        return self.out(x)

class CorrectModel(nn.Module):
    def __init__(self, n_layers=3):
        super().__init__()
        # CORRECT: ModuleList registers each layer
        self.layers = nn.ModuleList([nn.Linear(8, 8) for _ in range(n_layers)])
        self.out = nn.Linear(8, 2)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x).relu()
        return self.out(x)

buggy   = BuggyModel()
correct = CorrectModel()

buggy_params   = sum(p.numel() for p in buggy.parameters())
correct_params = sum(p.numel() for p in correct.parameters())

print(f"BuggyModel   parameter count: {buggy_params}")
print(f"CorrectModel parameter count: {correct_params}")
print(f"(missing {correct_params - buggy_params} parameters in buggy model!)")

# device test
correct.to(device)
print(f"\nAfter .to(device), CorrectModel layers[0].weight.device: {correct.layers[0].weight.device}")
buggy.to(device)
print(f"After .to(device), BuggyModel   layers[0].weight.device: {buggy.layers[0].weight.device}")
print("Note: BuggyModel layers stay on CPU even after .to(device)")

## 3. Buffers vs Parameters

Not everything in a model should be a learnable parameter. Running statistics in BatchNorm, positional embeddings, and the causal mask in transformers are tensors that:
- Need to move with the model when `.to(device)` is called
- Need to be saved/loaded with `state_dict()`
- Must NOT receive gradients or optimizer updates

`register_buffer` handles exactly this case. A buffer is tracked by the module machinery but not included in `parameters()`.

In [ ]:
class MovingAverageLayer(nn.Module):
    def __init__(self, size, alpha=0.9):
        super().__init__()
        self.alpha = alpha
        # learnable scale -- updated by optimizer
        self.scale = nn.Parameter(torch.ones(size))
        # running mean -- updated manually, not by optimizer
        self.register_buffer("running_mean", torch.zeros(size))
        self.register_buffer("running_std",  torch.ones(size))

    def forward(self, x):
        if self.training:
            # update running stats (no gradient tracking needed)
            with torch.no_grad():
                self.running_mean = self.alpha * self.running_mean + (1 - self.alpha) * x.mean(0)
                self.running_std  = self.alpha * self.running_std  + (1 - self.alpha) * x.std(0)
        return self.scale * (x - self.running_mean) / (self.running_std + 1e-5)

layer = MovingAverageLayer(4)

print("Parameters (get gradients, updated by optimizer):")
for name, p in layer.named_parameters():
    print(f"  {name}: {p.shape}")

print("\nBuffers (move with .to(), saved in state_dict, NO optimizer update):")
for name, b in layer.named_buffers():
    print(f"  {name}: {b.shape}")

sd = layer.state_dict()
print(f"\nstate_dict keys: {list(sd.keys())}")

# buffers move with .to()
layer.to(device)
print(f"\nAfter .to(device):")
print(f"  scale.device:        {layer.scale.device}")
print(f"  running_mean.device: {layer.running_mean.device}")

## 4. Hooks -- Debugging Activations and Gradients

Hooks let you intercept the forward or backward pass without modifying the model source code. They are the right tool for:
- Debugging exploding/vanishing activations
- Visualising which neurons activate on a given input
- Implementing gradient-based attribution (GradCAM)
- Modifying activations for adversarial attacks or probing

Three hook types: `register_forward_hook`, `register_backward_hook`, `register_forward_pre_hook`.

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
)

activation_stats = {}
gradient_stats   = {}

# Forward hook: record activation statistics
def forward_hook(name):
    def hook(module, input, output):
        activation_stats[name] = {
            "mean": output.detach().mean().item(),
            "std":  output.detach().std().item(),
            "frac_dead": (output.detach() == 0).float().mean().item()
        }
    return hook

# Backward hook: record gradient statistics
def backward_hook(name):
    def hook(module, grad_input, grad_output):
        g = grad_output[0]
        if g is not None:
            gradient_stats[name] = {
                "norm":    g.norm().item(),
                "max_abs": g.abs().max().item(),
            }
    return hook

# Register hooks
handles = []
for i, layer in enumerate(model):
    handles.append(layer.register_forward_hook(forward_hook(f"layer_{i}")))
    handles.append(layer.register_backward_hook(backward_hook(f"layer_{i}")))

# Run forward + backward
x = torch.randn(32, 8)
loss = model(x).sum()
loss.backward()

# Report
print("Activation statistics:")
for name, stats in activation_stats.items():
    print(f"  {name}: mean={stats['mean']:.3f}  std={stats['std']:.3f}  dead_frac={stats['frac_dead']:.2f}")

print("\nGradient statistics:")
for name, stats in gradient_stats.items():
    print(f"  {name}: norm={stats['norm']:.3f}  max_abs={stats['max_abs']:.3f}")

# Always remove hooks when done -- they accumulate if you re-run the cell
for h in handles:
    h.remove()
print("\nHooks removed.")

## 5. state_dict, load_state_dict, and strict=False

The `state_dict()` is an `OrderedDict` of parameter and buffer tensors keyed by their module path (e.g., `'encoder.layer.0.weight'`). `load_state_dict()` matches keys exactly by default (`strict=True`). Setting `strict=False` is essential for:
- Loading a pretrained backbone into a model with additional heads
- Resuming training after adding a new module
- Transfer learning where you intentionally skip some layers

In [ ]:
# Base model: pretrained-like
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, 8)

    def forward(self, x):
        return self.fc2(self.fc1(x).relu())

# Extended model: encoder + new classification head
class ExtendedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.head    = nn.Linear(8, 10)  # new -- not in pretrained state_dict

    def forward(self, x):
        return self.head(self.encoder(x))

# Simulate a pretrained checkpoint (only encoder weights)
encoder = Encoder()
pretrained_sd = {"encoder." + k: v for k, v in encoder.state_dict().items()}

# Load with strict=True -- would fail because 'head' is missing
extended = ExtendedModel()
try:
    extended.load_state_dict(pretrained_sd, strict=True)
except RuntimeError as e:
    print(f"strict=True fails (expected): {str(e)[:120]}...")

# Load with strict=False -- loads what matches, ignores the rest
missing, unexpected = extended.load_state_dict(pretrained_sd, strict=False)
print(f"\nstrict=False:")
print(f"  missing keys:    {missing}")
print(f"  unexpected keys: {unexpected}")
print("Encoder weights loaded; head is randomly initialised.")

## 6. train() vs eval() -- What Actually Changes

`model.train()` and `model.eval()` set a flag that is checked by specific layers. It is **not** a global flag that disables gradient computation -- you still need `torch.no_grad()` for that during inference.

Layers that behave differently in train vs eval:
- **BatchNorm**: train uses batch statistics and updates running_mean/running_var; eval uses the stored running statistics
- **Dropout**: train randomly zeros units; eval is a no-op (identity)
- **LayerNorm, GroupNorm**: unaffected by train/eval mode

In [ ]:
# Demonstrate the difference
bn = nn.BatchNorm1d(4)
dropout = nn.Dropout(p=0.5)

x = torch.randn(16, 4)

# Training mode
bn.train()
dropout.train()
out_bn_train = bn(x)
out_do_train = dropout(torch.ones(16, 4))

# Eval mode
bn.eval()
dropout.eval()
out_bn_eval = bn(x)          # uses running stats instead of batch stats
out_do_eval = dropout(torch.ones(16, 4))  # no dropout

print("BatchNorm train output mean:", out_bn_train.mean(0).detach())
print("BatchNorm eval  output mean:", out_bn_eval.mean(0).detach())

print(f"\nDropout train -- fraction zeroed: {(out_do_train == 0).float().mean():.2f}  (expected ~0.5)")
print(f"Dropout eval  -- fraction zeroed: {(out_do_eval  == 0).float().mean():.2f}  (expected 0.0)")

# IMPORTANT: model.eval() does NOT disable gradient tracking
model_eval = nn.Linear(4, 2)
model_eval.eval()
x_req = torch.randn(4, requires_grad=True)
out = model_eval(x_req)
print(f"\nAfter model.eval(): out.requires_grad = {out.requires_grad}")
print("You still need torch.no_grad() for memory efficiency in inference!")

## 7. Freezing Layers with requires_grad_(False)

Freezing is the standard pattern for fine-tuning: load a pretrained model, freeze the backbone, and only train the new head. `requires_grad_(False)` sets the flag recursively for a module and all its children. The frozen layers still participate in the forward pass (you get their representations) but contribute no gradient and no optimizer update.

In [ ]:
class FinetuneModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.head = nn.Linear(64, 10)

    def forward(self, x):
        return self.head(self.backbone(x))

model = FinetuneModel()

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Before freeze: trainable params = {count_trainable(model)}")

# Freeze backbone
model.backbone.requires_grad_(False)

print(f"After freeze:  trainable params = {count_trainable(model)}")
print(f"  backbone params frozen: {not any(p.requires_grad for p in model.backbone.parameters())}")
print(f"  head params trainable:  {all(p.requires_grad for p in model.head.parameters())}")

# Only head parameters should go to the optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
print(f"\nOptimizer param groups: {len(optimizer.param_groups)}")
print(f"Optimizer manages {sum(p.numel() for g in optimizer.param_groups for p in g['params'])} parameters")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| __setattr__ registration | Assigning nn.Parameter or nn.Module registers it; plain tensors are invisible |
| ModuleList vs list | Plain Python list does not register submodules -- silent bug |
| register_buffer | Tensor that moves with .to(), appears in state_dict, but not in parameters() |
| Hooks | Intercept forward/backward without modifying model code; always remove when done |
| state_dict | OrderedDict of all parameters + buffers; use strict=False for partial loading |
| train() vs eval() | Changes BatchNorm and Dropout behavior; does NOT affect gradient tracking |
| requires_grad_(False) | Freeze a module recursively; use filter() to pass only trainable params to optimizer |

### Common Pitfalls
- Storing submodules in a plain list -- they are invisible to `.to()`, optimizers, and `state_dict()`
- Assigning a plain `torch.Tensor` instead of `nn.Parameter` -- not updated by optimizer
- Calling `model.eval()` without `torch.no_grad()` in inference -- still builds the graph
- Forgetting `h.remove()` on hooks -- hooks accumulate and slow down every forward pass
- Loading a partial state_dict with `strict=True` -- RuntimeError you could avoid with `strict=False`

### Next: Notebook 04 -- Training Loop Patterns